# Level 4.14 — Tokenizer Evaluation Workshop

Compare a deterministic tiny word tokenizer with UTF-8 byte tokenization on the same evaluation slices. Record token counts, unknown behavior, and worst cases before making a recommendation.

**Expected completion evidence:** the final cell prints `PASS: l04-14 tokenizer evaluation checks`.


In [ ]:
import re

TRAIN_TEXT = ["tiny models learn", "models predict tokens", "players played"]
EVAL_CASES = {
    "familiar": "tiny models predict",
    "rare_name": "quokka predicts",
    "emoji": "tiny 🙂",
    "multilingual": "모델 learns",
}

def word_tokens(text):
    return re.findall(r"\w+|[^\w\s]", text.lower(), flags=re.UNICODE)

WORD_VOCAB = {"<unk>", *(piece for text in TRAIN_TEXT for piece in word_tokens(text))}

def word_encode(text):
    return [piece if piece in WORD_VOCAB else "<unk>" for piece in word_tokens(text)]

def byte_encode(text):
    return list(text.encode("utf-8"))

def unknown_rate(pieces):
    return 0.0 if not pieces else sum(piece == "<unk>" for piece in pieces) / len(pieces)


In [ ]:
rows = []
for name, text in EVAL_CASES.items():
    word_pieces = word_encode(text)
    byte_ids = byte_encode(text)
    rows.append({
        "slice": name,
        "text": text,
        "word_tokens": len(word_pieces),
        "word_unknown_rate": round(unknown_rate(word_pieces), 3),
        "byte_tokens": len(byte_ids),
        "word_pieces": word_pieces,
    })

for row in rows:
    print(row)


## Break and debug

Temporarily evaluate only the `familiar` slice. Notice how that restricted test can make the tiny word vocabulary look stronger than it is. Restore all slices, inspect the exact pieces for the worst unknown-rate case, and explain the changed conclusion.


In [ ]:
assert unknown_rate(word_encode(EVAL_CASES["familiar"])) == 0.0
assert unknown_rate(word_encode(EVAL_CASES["rare_name"])) > 0.0
assert bytes(byte_encode("Token🙂")).decode("utf-8") == "Token🙂"
assert rows == [dict(row) for row in rows]  # deterministic in-memory result
print("PASS: l04-14 tokenizer evaluation checks")


## Decision note

Write 4–8 sentences that name the target text distribution, compare at least two measured criteria, identify one worst-case slice, and state one limitation this small evaluation does not answer.
